Análisis posterior a la prueba. Este cuaderno examina los flujos mal clasificados del conjunto de prueba después de haber fijado el modelo, el umbral (0.87) y todos los hiperparámetros. Se usa únicamente para interpretar los errores; ningún resultado de este cuaderno se utilizó para ajustar, seleccionar ni modificar el modelo.

## Alcance y fuentes

Este cuaderno solo lee artefactos existentes. No reentrena, no vuelve a predecir y no cambia el umbral, la semilla ni ningún hiperparámetro.

- **Modelo congelado:** `models/xgboost/`, cargado con `load_xgboost_model()`, igual que en `06_evaluacion_final.ipynb` y `07_importancia.ipynb`. Se usa para verificar los metadatos y recalcular el ranking por gain.
- **Partición guardada:** `data/processed/xgboost/X_*.csv` e `y_*.csv`, leídos igual que en `02_modelo.ipynb` y `06_evaluacion_final.ipynb`.
- **Probabilidades de prueba:** `reports/xgboost/probabilidades_test.csv`, guardadas por `06_evaluacion_final.ipynb`.
- **Etiqueta original, bloque temporal e índice original de cada flujo:** se reconstruyen con `cargar_dataset_limpio` y `train_test_split_by_flow(..., return_meta=True)`, igual que en el 06. Se verifica que la reconstrucción coincide fila por fila con `X_test.csv` e `y_test.csv`.
- **Subconjunto común con la GRU:** `results/gru/03_evaluacion_test/idx_test_comun.npy`.
- **Ranking de features por gain:** `reports/xgboost/importancia_gain.csv`, generado por el 07 y contrastado con el modelo cargado.

La base del análisis es el subconjunto común. El test completo se reporta al lado cuando una cifra difiere.

Las tablas se guardan como CSV en `reports/xgboost/analisis_errores/`, la única carpeta en la que escribe este cuaderno. Si falta algún artefacto, el cuaderno se detiene: no regenera nada.

In [1]:
import json
import sys
from pathlib import Path

# Raíz del repo = primera carpeta hacia arriba que contenga src/.
RAIZ_REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ_REPO) not in sys.path:
    sys.path.append(str(RAIZ_REPO))

import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import Markdown, display
from sklearn.metrics import confusion_matrix

from src.models.xgboost.model import (
    ARCHIVO_METADATOS,
    ARCHIVO_MODELO,
    MODEL_DIR,
    UMBRAL_FINAL,
    apply_threshold,
    load_xgboost_model,
)
from src.preprocessing.clean import (
    COLUMNA_ETIQUETA,
    ETIQUETA_BENIGNA,
    ETIQUETAS_LOW_AND_SLOW,
    cargar_dataset_limpio,
    reportar_duplicados_de_features,
    train_test_split_by_flow,
)

DATA_DIR = RAIZ_REPO / "data" / "processed" / "xgboost"
REPORTES_XGB_DIR = RAIZ_REPO / "reports" / "xgboost"
RUTA_PROBA_TEST = REPORTES_XGB_DIR / "probabilidades_test.csv"
RUTA_RESULTADOS_TEST = REPORTES_XGB_DIR / "resultados_test.json"
RUTA_IMPORTANCIA = REPORTES_XGB_DIR / "importancia_gain.csv"
RUTA_DATASET_LIMPIO = RAIZ_REPO / "data" / "processed" / "wednesday_low_and_slow_clean.csv"
RUTA_IDX_COMUN = RAIZ_REPO / "results" / "gru" / "03_evaluacion_test" / "idx_test_comun.npy"
SALIDA_DIR = REPORTES_XGB_DIR / "analisis_errores"

ARCHIVOS_REQUERIDOS = [
    MODEL_DIR / ARCHIVO_MODELO,
    MODEL_DIR / ARCHIVO_METADATOS,
    *(DATA_DIR / f"{n}.csv" for n in ("X_train", "y_train", "X_val", "y_val", "X_test", "y_test")),
    RUTA_PROBA_TEST,
    RUTA_RESULTADOS_TEST,
    RUTA_IMPORTANCIA,
    RUTA_DATASET_LIMPIO,
    RUTA_IDX_COMUN,
]
faltantes = [str(r.relative_to(RAIZ_REPO)) for r in ARCHIVOS_REQUERIDOS if not r.exists()]
if faltantes:
    raise FileNotFoundError(
        "Faltan artefactos guardados; este cuaderno no los regenera y se detiene: " + ", ".join(faltantes)
    )

SALIDA_DIR.mkdir(parents=True, exist_ok=True)


def guardar(tabla: pd.DataFrame, nombre: str, index: bool = False) -> None:
    """Guarda una tabla como CSV en SALIDA_DIR, la única carpeta en la que escribe este cuaderno."""
    ruta = SALIDA_DIR / f"{nombre}.csv"
    tabla.to_csv(ruta, index=index)
    print(f"Tabla guardada en {ruta.relative_to(RAIZ_REPO)}")


print(f"Python:        {sys.executable}")
print(f"Raíz del repo: {RAIZ_REPO}")
print(f"xgboost {xgb.__version__} | pandas {pd.__version__}")
print(f"Artefactos requeridos presentes: {len(ARCHIVOS_REQUERIDOS)} de {len(ARCHIVOS_REQUERIDOS)}")
print(f"Salidas: {SALIDA_DIR.relative_to(RAIZ_REPO)}")

Python:        C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids\venv\Scripts\python.exe
Raíz del repo: C:\Users\Andy Ortega\Progras\Proyecto de Graduación\low-and-slow-ids
xgboost 3.3.0 | pandas 3.0.5
Artefactos requeridos presentes: 13 de 13
Salidas: reports\xgboost\analisis_errores


## Modelo congelado

Se carga solo para comprobar que el umbral guardado en los metadatos es el umbral final del módulo y que `best_iteration` coincide. Las probabilidades no se vuelven a calcular.

In [2]:
modelo, metadatos = load_xgboost_model()
assert metadatos["umbral"] == UMBRAL_FINAL, f"Umbral de los metadatos {metadatos['umbral']} != UMBRAL_FINAL {UMBRAL_FINAL}"
assert metadatos["best_iteration"] == modelo.best_iteration, "best_iteration de los metadatos != modelo cargado"

print(f"Modelo: {MODEL_DIR.relative_to(RAIZ_REPO)} | umbral {UMBRAL_FINAL} | best_iteration {modelo.best_iteration} | "
      f"scale_pos_weight {metadatos['scale_pos_weight']:.4f}")
print("Hiperparámetros congelados:", json.dumps(metadatos["hiperparametros"], ensure_ascii=False))

Modelo: models\xgboost | umbral 0.87 | best_iteration 68 | scale_pos_weight 36.7313
Hiperparámetros congelados: {"learning_rate": 0.3, "max_depth": 4, "min_child_weight": 1, "n_estimators": 1000, "early_stopping_rounds": 50, "eval_metric": "aucpr", "tree_method": "hist", "random_state": 42, "n_jobs": -1}


## Carga de la partición guardada y de las probabilidades

Igual que en el 02 y el 06: `X` como DataFrame y `y` como arreglo 1D de la columna `label_binaria`. Train y val se cargan solo para la verificación de duplicados exactos de la sección 5.

In [3]:
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")["label_binaria"].to_numpy()
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
y_val = pd.read_csv(DATA_DIR / "y_val.csv")["label_binaria"].to_numpy()
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_test = pd.read_csv(DATA_DIR / "y_test.csv")["label_binaria"].to_numpy()

COLUMNAS = list(X_test.columns)
assert list(X_train.columns) == list(X_val.columns) == COLUMNAS, "Columnas distintas entre train, val y test"
assert COLUMNAS == modelo.get_booster().feature_names, "Las columnas de X_test no son las features del modelo"

for nombre, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{nombre:>5}: X {X.shape} | ataques {int(y.sum()):,} | benignos {int((y == 0).sum()):,}")

guardadas = pd.read_csv(RUTA_PROBA_TEST, float_precision="round_trip")
assert len(guardadas) == len(y_test), f"El archivo tiene {len(guardadas):,} filas, test tiene {len(y_test):,}"
assert np.array_equal(guardadas["label_binaria"].to_numpy(), y_test), "Las probabilidades no corresponden a y_test"
proba = guardadas["probabilidad"].to_numpy(dtype=np.float64)
print(f"\nProbabilidades leídas de {RUTA_PROBA_TEST.relative_to(RAIZ_REPO)}: {len(proba):,} flujos")

train: X (310529, 55) | ataques 8,230 | benignos 302,299
  val: X (66541, 55) | ataques 1,884 | benignos 64,657
 test: X (66544, 55) | ataques 1,175 | benignos 65,369

Probabilidades leídas de reports\xgboost\probabilidades_test.csv: 66,544 flujos


### Etiqueta original, bloque temporal e índice original

`X_test.csv` no guarda la etiqueta original (`Label`), el bloque temporal ni el índice del flujo en el dataset limpio. Se reconstruyen con el mismo procedimiento del 06 y se exige que las features reconstruidas coincidan con `X_test.csv` fila por fila. El índice original es la etiqueta de fila en `wednesday_low_and_slow_clean.csv`.

In [4]:
df_limpio = cargar_dataset_limpio(RUTA_DATASET_LIMPIO)
_, _, X_test_rec, _, _, y_test_rec, _, _, meta_test = train_test_split_by_flow(df_limpio, return_meta=True)
del df_limpio

assert np.array_equal(y_test_rec.to_numpy(), y_test), "La etiqueta binaria reconstruida no coincide con y_test"
assert np.allclose(X_test_rec[COLUMNAS].to_numpy(dtype=np.float64), X_test.to_numpy(dtype=np.float64),
                   rtol=1e-12, atol=0.0), "Las features reconstruidas no coinciden con X_test.csv"

etiqueta_original = meta_test[COLUMNA_ETIQUETA].to_numpy()
indice_test = meta_test.index.to_numpy()
bloque_test = X_test_rec["bloque"].to_numpy()
del X_test_rec, y_test_rec
assert len(np.unique(indice_test)) == len(indice_test), "El índice reconstruido no es único"
assert np.array_equal(etiqueta_original != ETIQUETA_BENIGNA, y_test == 1), "Label y label_binaria no coinciden"

NOMBRE_TIPO = {ETIQUETA_BENIGNA: "benigno", "DoS slowloris": "Slowloris", "DoS Slowhttptest": "Slowhttptest"}
assert set(NOMBRE_TIPO) == {ETIQUETA_BENIGNA, *ETIQUETAS_LOW_AND_SLOW}, "Etiquetas distintas a las de clean.py"
tipo = pd.Series(etiqueta_original).map(NOMBRE_TIPO).to_numpy()
TIPOS_ATAQUE = [NOMBRE_TIPO[t] for t in ETIQUETAS_LOW_AND_SLOW]
print("\nReconstrucción idéntica a X_test.csv / y_test.csv. Flujos por tipo en test completo:")
print(pd.Series(tipo).value_counts().to_string())

Fronteras de bloque (outlier_quantile=0.0, desempate=csv):
  Slowloris:    [2017-07-05 09:48:00 .. 2017-07-05 10:11:00]  (5790 filas)
  Slowhttptest: [2017-07-05 10:15:00 .. 2017-07-05 10:37:00]  (5499 filas)

Composición por bloque:
  Bloque 1: 53375 filas | Timestamp [2017-07-05 08:42:00 .. 2017-07-05 09:47:00] | ataques: 0 (0.00%)
  Bloque 2: 30362 filas | Timestamp [2017-07-05 09:48:00 .. 2017-07-05 10:11:00] | ataques: 5790 (19.07%)
  Bloque 3: 3578 filas | Timestamp [2017-07-05 10:12:00 .. 2017-07-05 10:14:00] | ataques: 0 (0.00%)
  Bloque 4: 33203 filas | Timestamp [2017-07-05 10:15:00 .. 2017-07-05 10:37:00] | ataques: 5499 (16.56%)


  Bloque 5: 323096 filas | Timestamp [2017-07-05 10:38:00 .. 2017-07-05 17:10:00] | ataques: 0 (0.00%)



Train: 310529 filas | Val: 66541 filas | Test: 66544 filas
Proporción de ataques -> train: 0.0265, val: 0.0283, test: 0.0177

Reconstrucción idéntica a X_test.csv / y_test.csv. Flujos por tipo en test completo:


benigno         65369
Slowloris         915
Slowhttptest      260


In [5]:
idx_comun = np.load(RUTA_IDX_COMUN, allow_pickle=False)
assert idx_comun.ndim == 1 and len(np.unique(idx_comun)) == len(idx_comun), "Lista de filas comunes inválida"
assert len(np.setdiff1d(idx_comun, indice_test)) == 0, "Hay etiquetas de la lista que no están en el índice de test"
mascara_comun = np.isin(indice_test, idx_comun)
assert mascara_comun.sum() == len(idx_comun)

AMBITOS = {"subconjunto común": mascara_comun, "test completo": np.ones(len(y_test), dtype=bool)}
for ambito, m in AMBITOS.items():
    print(f"{ambito:>17}: {int(m.sum()):,} flujos | {int(y_test[m].sum()):,} ataques | {int((y_test[m] == 0).sum()):,} benignos")

subconjunto común: 66,454 flujos | 1,164 ataques | 65,290 benignos
    test completo: 66,544 flujos | 1,175 ataques | 65,369 benignos


## 1. Matriz de confusión y lista de errores

La predicción se obtiene con `apply_threshold` y el umbral final del módulo. Las matrices se contrastan con las que guardó el 06 en `reports/xgboost/resultados_test.json`.

In [6]:
y_pred = apply_threshold(proba)  # UMBRAL_FINAL por defecto
es_tp = (y_test == 1) & (y_pred == 1)
es_fn = (y_test == 1) & (y_pred == 0)
es_fp = (y_test == 0) & (y_pred == 1)
es_tn = (y_test == 0) & (y_pred == 0)
es_error = es_fn | es_fp
resultado = np.select([es_tp, es_fn, es_fp], ["TP", "FN", "FP"], default="TN")

filas = {}
for ambito, m in AMBITOS.items():
    tn, fp, fn, tp = confusion_matrix(y_test[m], y_pred[m], labels=[0, 1]).ravel()
    filas[ambito] = {"flujos": int(m.sum()), "ataques": int(y_test[m].sum()), "benignos": int((y_test[m] == 0).sum()),
                     "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}
matriz_confusion = pd.DataFrame(filas).T

resultados_06 = json.loads(RUTA_RESULTADOS_TEST.read_text(encoding="utf-8"))
for ambito, clave in [("test completo", "test_completo"), ("subconjunto común", "subconjunto_comun_gru")]:
    mc = resultados_06[clave]["metricas_umbral_principal"]["matriz_confusion"]
    guardada = {"TN": mc["verdaderos_negativos"], "FP": mc["falsos_positivos"],
                "FN": mc["falsos_negativos"], "TP": mc["verdaderos_positivos"]}
    obtenida = matriz_confusion.loc[ambito, ["TN", "FP", "FN", "TP"]].to_dict()
    assert obtenida == guardada, f"{ambito}: matriz recalculada {obtenida} != resultados_test.json {guardada}"

print(f"Matriz de confusión, umbral {UMBRAL_FINAL} (coincide con resultados_test.json del 06):")
print(matriz_confusion.to_string())
print(f"\nErrores fuera del subconjunto común: FN {int((es_fn & ~mascara_comun).sum())} | "
      f"FP {int((es_fp & ~mascara_comun).sum())}")
guardar(matriz_confusion.rename_axis("ambito"), "matriz_confusion", index=True)

Matriz de confusión, umbral 0.87 (coincide con resultados_test.json del 06):
                   flujos  ataques  benignos     TN  FP  FN    TP
subconjunto común   66454     1164     65290  65282   8  11  1153
test completo       66544     1175     65369  65361   8  11  1164

Errores fuera del subconjunto común: FN 0 | FP 0
Tabla guardada en reports\xgboost\analisis_errores\matriz_confusion.csv


### Las 12 features con mayor gain

Se toman de `importancia_gain.csv` (07). Antes se comprueba que el orden coincide con el gain recalculado sobre los árboles que usa la predicción del modelo cargado (`booster[: best_iteration + 1]`), con el mismo criterio de orden del 07.

In [7]:
N_TOP = 12
importancia = pd.read_csv(RUTA_IMPORTANCIA).sort_values("posicion")
booster_usado = modelo.get_booster()[: modelo.best_iteration + 1]
gain_modelo = booster_usado.get_score(importance_type="gain")
ranking_modelo = (pd.DataFrame({"feature": COLUMNAS, "gain": [gain_modelo.get(c, 0.0) for c in COLUMNAS]})
                  .sort_values(["gain", "feature"], ascending=[False, True])["feature"].tolist())
assert ranking_modelo == importancia["feature"].tolist(), "El ranking de importancia_gain.csv no coincide con el modelo"

TOP = importancia["feature"].head(N_TOP).tolist()
print(f"Top {N_TOP} por gain (coincide con el modelo cargado):")
print(importancia.head(N_TOP)[["posicion", "feature", "prop_gain", "prop_gain_acumulada"]]
      .to_string(index=False, float_format="{:.4f}".format))

Top 12 por gain (coincide con el modelo cargado):
 posicion                     feature  prop_gain  prop_gain_acumulada
        1      Bwd Packet Length Mean     0.5631               0.5631
        2               Fwd IAT Total     0.1453               0.7084
        3 Total Length of Fwd Packets     0.0566               0.7650
        4                Fwd IAT Mean     0.0528               0.8179
        5       Fwd Packet Length Min     0.0461               0.8640
        6        min_seg_size_forward     0.0339               0.8979
        7              PSH Flag Count     0.0102               0.9081
        8                    Idle Max     0.0097               0.9178
        9       Fwd Packet Length Max     0.0081               0.9259
       10      Packet Length Variance     0.0080               0.9340
       11                 Fwd IAT Max     0.0079               0.9418
       12      Fwd Packet Length Mean     0.0064               0.9483


In [8]:
def tabla_flujos(mascara: np.ndarray, columnas: list[str]) -> pd.DataFrame:
    """Una fila por flujo de la máscara: identificación, resultado, probabilidad, bloque y las columnas pedidas."""
    t = pd.DataFrame({
        "indice_original": indice_test[mascara],
        "etiqueta_real": etiqueta_original[mascara],
        "tipo": tipo[mascara],
        "resultado": resultado[mascara],
        "probabilidad": proba[mascara],
        "bloque": bloque_test[mascara],
        "en_subconjunto_comun": mascara_comun[mascara],
    })
    for c in columnas:
        t[c] = X_test.loc[mascara, c].to_numpy()
    return t.sort_values(["resultado", "tipo", "probabilidad"], ignore_index=True)


errores = tabla_flujos(es_error, TOP)
print(f"Errores en test completo: {len(errores)} (FN {int(es_fn.sum())}, FP {int(es_fp.sum())})")
with pd.option_context("display.width", 300, "display.max_columns", 40):
    print(errores.to_string(float_format=lambda v: f"{v:.6g}"))
guardar(errores, "errores")

Errores en test completo: 19 (FN 11, FP 8)


    indice_original     etiqueta_real          tipo resultado  probabilidad  bloque  en_subconjunto_comun  Bwd Packet Length Mean  Fwd IAT Total  Total Length of Fwd Packets  Fwd IAT Mean  Fwd Packet Length Min  min_seg_size_forward  PSH Flag Count  Idle Max  Fwd Packet Length Max  Packet Length Variance  Fwd IAT Max  Fwd Packet Length Mean
0            119790  DoS Slowhttptest  Slowhttptest        FN    0.00418557       4                  True                     535            803                         1454           803                      6                    20               0         0                   1448                  508809          803                     727
1            119789  DoS Slowhttptest  Slowhttptest        FN     0.0105242       4                  True                       0            737                         1989           737                      6                    20               0         0                   1983             1.30284e+06        

## 2. Falsos negativos

Clasificación operativa de cada FN según la distancia de su probabilidad al umbral:

- **Error de umbral:** la probabilidad está a 0.1 o menos por debajo del umbral.
- **Error sin señal:** la probabilidad está a más de 0.1 por debajo del umbral.

La etiqueta solo describe esa distancia; no dice por qué el modelo asignó esa probabilidad.

In [9]:
VENTANA_UMBRAL = 0.1
LIMITE_UMBRAL = round(UMBRAL_FINAL - VENTANA_UMBRAL, 10)

fn = tabla_flujos(es_fn, ["Total Length of Fwd Packets"])
fn["categoria"] = np.where(fn["probabilidad"] >= LIMITE_UMBRAL, "error de umbral", "error sin señal")
fn["distancia_al_umbral"] = UMBRAL_FINAL - fn["probabilidad"]

print("FN por tipo de ataque:")
print(fn["tipo"].value_counts().reindex(TIPOS_ATAQUE, fill_value=0).to_string())

print("\nProbabilidades de los FN, de menor a mayor:")
print(fn.sort_values("probabilidad")[["indice_original", "tipo", "probabilidad", "distancia_al_umbral", "categoria"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

TRAMOS = [0.0, VENTANA_UMBRAL, LIMITE_UMBRAL, UMBRAL_FINAL]
fn["tramo_probabilidad"] = pd.cut(fn["probabilidad"], bins=TRAMOS, right=False).astype(str)
tramos_fn = pd.crosstab(fn["tramo_probabilidad"], fn["tipo"], margins=True, margins_name="total")
print("\nFN por tramo de probabilidad (intervalos [a, b)):")
print(tramos_fn.to_string())

categorias_fn = pd.crosstab(fn["categoria"], fn["tipo"], margins=True, margins_name="total")
print(f"\nFN por categoría (límite {LIMITE_UMBRAL}):")
print(categorias_fn.to_string())

guardar(fn, "falsos_negativos")
guardar(tramos_fn.rename_axis(index="tramo_probabilidad", columns=None), "falsos_negativos_por_tramo", index=True)
guardar(categorias_fn.rename_axis(index="categoria", columns=None), "falsos_negativos_por_categoria", index=True)

FN por tipo de ataque:
tipo
Slowloris       8
Slowhttptest    3

Probabilidades de los FN, de menor a mayor:
 indice_original         tipo  probabilidad  distancia_al_umbral       categoria
           82946    Slowloris        0.0000               0.8700 error sin señal
           82962    Slowloris        0.0001               0.8699 error sin señal
          119790 Slowhttptest        0.0042               0.8658 error sin señal
           82965    Slowloris        0.0067               0.8633 error sin señal
           82963    Slowloris        0.0071               0.8629 error sin señal
           82959    Slowloris        0.0073               0.8627 error sin señal
          119789 Slowhttptest        0.0105               0.8595 error sin señal
           82964    Slowloris        0.0239               0.8461 error sin señal
          116669 Slowhttptest        0.3268               0.5432 error sin señal
           82819    Slowloris        0.8608               0.0092 error de umbral


Tabla guardada en reports\xgboost\analisis_errores\falsos_negativos.csv
Tabla guardada en reports\xgboost\analisis_errores\falsos_negativos_por_tramo.csv
Tabla guardada en reports\xgboost\analisis_errores\falsos_negativos_por_categoria.csv


### Payload forward igual a cero

Cantidad de flujos con `Total Length of Fwd Packets` igual a 0 en los FN y, como referencia, en los ataques detectados de cada tipo y en los verdaderos negativos.

In [10]:
payload_cero = (X_test["Total Length of Fwd Packets"] == 0).to_numpy()
GRUPOS_PAYLOAD = {
    "FN (ambos tipos)": es_fn,
    "FN Slowloris": es_fn & (tipo == "Slowloris"),
    "FN Slowhttptest": es_fn & (tipo == "Slowhttptest"),
    "TP Slowloris": es_tp & (tipo == "Slowloris"),
    "TP Slowhttptest": es_tp & (tipo == "Slowhttptest"),
    "TN": es_tn,
}
filas = []
for ambito, m_amb in AMBITOS.items():
    for grupo, m in GRUPOS_PAYLOAD.items():
        n = int((m & m_amb).sum())
        k = int((m & m_amb & payload_cero).sum())
        filas.append({"ambito": ambito, "grupo": grupo, "flujos": n, "payload_fwd_cero": k,
                      "proporcion": k / n if n else np.nan})
payload = pd.DataFrame(filas)
tabla_payload = payload.pivot(index="grupo", columns="ambito", values=["flujos", "payload_fwd_cero", "proporcion"])
tabla_payload = tabla_payload.reindex(list(GRUPOS_PAYLOAD))
for col in ("flujos", "payload_fwd_cero"):
    tabla_payload[col] = tabla_payload[col].astype(int)
with pd.option_context("display.width", 200):
    print(tabla_payload.to_string(float_format=lambda v: f"{v:.4f}"))
difiere = payload.pivot(index="grupo", columns="ambito", values="payload_fwd_cero")
difiere = difiere[difiere["subconjunto común"] != difiere["test completo"]]
print("\nGrupos donde el conteo difiere entre ámbitos: " + (", ".join(difiere.index) if len(difiere) else "ninguno"))
guardar(payload, "payload_fwd_cero")

                            flujos                payload_fwd_cero                      proporcion              
ambito           subconjunto común test completo subconjunto común test completo subconjunto común test completo
grupo                                                                                                           
FN (ambos tipos)                11            11                 6             6            0.5455        0.5455
FN Slowloris                     8             8                 5             5            0.6250        0.6250
FN Slowhttptest                  3             3                 1             1            0.3333        0.3333
TP Slowloris                   906           907               284           285            0.3135        0.3142
TP Slowhttptest                247           257                56            65            0.2267        0.2529
TN                           65282         65361              7080          7096            0.10

## 3. Falsos positivos

Probabilidad, conteos de paquetes, duración, payload forward, `min_seg_size_forward`, valores de `Init_Win_bytes_*` y bloque temporal de cada FP.

In [11]:
COLUMNAS_FP = ["Total Fwd Packets", "Total Backward Packets", "Flow Duration", "Total Length of Fwd Packets",
               "min_seg_size_forward", "Init_Win_bytes_forward", "Init_Win_bytes_backward"]
assert set(COLUMNAS_FP) <= set(COLUMNAS), f"Faltan columnas: {set(COLUMNAS_FP) - set(COLUMNAS)}"

fp = tabla_flujos(es_fp, COLUMNAS_FP)
with pd.option_context("display.width", 300, "display.max_columns", 40):
    print(fp.drop(columns=["etiqueta_real", "tipo", "resultado"]).to_string(float_format=lambda v: f"{v:.6g}"))

resumen_fp = fp[["probabilidad", *COLUMNAS_FP]].agg(["min", "median", "max"]).T
resumen_fp["valores_distintos"] = fp[["probabilidad", *COLUMNAS_FP]].nunique()
print("\nRango de cada variable en los FP:")
print(resumen_fp.to_string(float_format=lambda v: f"{v:.6g}"))

print("\nFP por bloque temporal:")
print(fp["bloque"].value_counts().sort_index().to_string())
print(f"\nFP con payload forward igual a cero: {int((fp['Total Length of Fwd Packets'] == 0).sum())} de {len(fp)}")
print(f"FP con Total Backward Packets igual a cero: {int((fp['Total Backward Packets'] == 0).sum())} de {len(fp)}")

guardar(fp, "falsos_positivos")
guardar(resumen_fp.rename_axis("variable"), "falsos_positivos_resumen", index=True)

   indice_original  probabilidad  bloque  en_subconjunto_comun  Total Fwd Packets  Total Backward Packets  Flow Duration  Total Length of Fwd Packets  min_seg_size_forward  Init_Win_bytes_forward  Init_Win_bytes_backward
0           396766      0.997196       5                  True                  6                       0       31066281                            0                    40                   29200                       -1
1           116965      0.998301       4                  True                  7                       0       63147176                            0                    40                   29200                       -1
2           116953      0.998536       4                  True                  7                       0       63141277                            0                    40                   29200                       -1
3           396517      0.998536       5                  True                  7                       0       6309



Rango de cada variable en los FP:
                                    min     median         max  valores_distintos
probabilidad                   0.997196   0.998681    0.999258                  7
Total Fwd Packets                     6          7           7                  2
Total Backward Packets                0          0           0                  1
Flow Duration               3.10663e+07 6.3114e+07 6.31472e+07                  8
Total Length of Fwd Packets           0          0           0                  1
min_seg_size_forward                 40         40          40                  1
Init_Win_bytes_forward            29200      29200       29200                  1
Init_Win_bytes_backward              -1         -1          -1                  1

FP por bloque temporal:
bloque
1    2
4    2
5    4

FP con payload forward igual a cero: 8 de 8
FP con Total Backward Packets igual a cero: 8 de 8
Tabla guardada en reports\xgboost\analisis_errores\falsos_positivos.csv
Tabla

## 4. Comparación descriptiva con grupos de referencia

Para cada error y cada una de las 12 features con mayor gain se indica si el valor cae dentro del rango P5–P95 de dos grupos:

- **(a) Ataques detectados (TP) del mismo tipo.** Para un FN Slowloris, los Slowloris detectados; para un FN Slowhttptest, los Slowhttptest detectados. Los FP son benignos y no tienen tipo de ataque, así que se comparan con todos los ataques detectados.
- **(b) Verdaderos negativos (TN).**

Los percentiles se calculan sobre el subconjunto común. Los conteos también se recalculan con percentiles del test completo, y se indica si cambian.

> **Nota.** En varias de estas features los rangos P5–P95 son muy amplios, tanto en los benignos como en los ataques detectados; la tabla de anchos de abajo los compara feature por feature. Un valor dentro de un rango tan amplio aporta poca información. Esto es solo un conteo descriptivo y un criterio débil; **no es una medida de similitud** y no indica a qué grupo se parece un flujo.

In [12]:
CUANTILES = {"P5": 0.05, "P50": 0.50, "P95": 0.95}


def grupos_referencia(m_amb: np.ndarray) -> dict:
    return {
        "TP Slowloris": es_tp & (tipo == "Slowloris") & m_amb,
        "TP Slowhttptest": es_tp & (tipo == "Slowhttptest") & m_amb,
        "TP (ambos tipos)": es_tp & m_amb,
        "TN": es_tn & m_amb,
    }


def percentiles(m: np.ndarray) -> pd.DataFrame:
    sub = X_test.loc[m, TOP]
    return pd.DataFrame({nombre: sub.quantile(q) for nombre, q in CUANTILES.items()})


rangos = {amb: {g: percentiles(m) for g, m in grupos_referencia(m_amb).items()} for amb, m_amb in AMBITOS.items()}
filas = []
for amb, por_grupo in rangos.items():
    for g, r in por_grupo.items():
        n = int(grupos_referencia(AMBITOS[amb])[g].sum())
        for feat, fila in r.iterrows():
            filas.append({"ambito": amb, "grupo": g, "flujos": n, "feature": feat, **fila.to_dict()})
tabla_rangos = pd.DataFrame(filas)
guardar(tabla_rangos, "rangos_referencia")

ref = rangos["subconjunto común"]
print("Percentiles de referencia (subconjunto común):")
for g, r in ref.items():
    print(f"\n{g} (n = {int(grupos_referencia(mascara_comun)[g].sum()):,})")
    print(r.T.to_string(float_format=lambda v: f"{v:.6g}"))

anchos = pd.DataFrame({g: r["P95"] - r["P5"] for g, r in ref.items()})
print("\nAncho del rango P5–P95 por grupo (subconjunto común):")
print(anchos.to_string(float_format=lambda v: f"{v:.6g}"))
guardar(anchos.rename_axis("feature"), "anchos_rango_p5_p95", index=True)

Tabla guardada en reports\xgboost\analisis_errores\rangos_referencia.csv


Percentiles de referencia (subconjunto común):

TP Slowloris (n = 906)
     Bwd Packet Length Mean  Fwd IAT Total  Total Length of Fwd Packets  Fwd IAT Mean  Fwd Packet Length Min  min_seg_size_forward  PSH Flag Count  Idle Max  Fwd Packet Length Max  Packet Length Variance  Fwd IAT Max  Fwd Packet Length Mean
P5                        0              0                            0             0                      0                    32               0         0                      0                       0            0                       0
P50                       0       3.19e+07                            8   3.38211e+06                      0                    32               1         0                      8                 21.3333     1.28e+07                       4
P95                   241.5       1.06e+08                         2541         1e+08                    231                    40               1  5.14e+07                    239                 58322.2  

In [13]:
def posicion(valor: float, r: pd.Series) -> str:
    return "debajo de P5" if valor < r["P5"] else ("encima de P95" if valor > r["P95"] else "dentro")


def comparar(ambito: str) -> pd.DataFrame:
    """Una fila por (error, feature) con la posición frente a (a) y (b)."""
    filas = []
    for _, e in errores.iterrows():
        grupo_a = f"TP {e['tipo']}" if e["resultado"] == "FN" else "TP (ambos tipos)"
        for feat in TOP:
            v = e[feat]
            filas.append({
                "indice_original": e["indice_original"], "resultado": e["resultado"], "tipo": e["tipo"],
                "feature": feat, "valor": v, "grupo_a": grupo_a,
                "posicion_a": posicion(v, rangos[ambito][grupo_a].loc[feat]),
                "posicion_b_TN": posicion(v, rangos[ambito]["TN"].loc[feat]),
            })
    return pd.DataFrame(filas)


comparacion = {amb: comparar(amb) for amb in AMBITOS}


def conteo(c: pd.DataFrame) -> pd.DataFrame:
    return (c.assign(dentro_a=c["posicion_a"] == "dentro", dentro_b=c["posicion_b_TN"] == "dentro")
            .groupby(["indice_original", "resultado", "tipo", "grupo_a"], as_index=False)[["dentro_a", "dentro_b"]].sum()
            .rename(columns={"dentro_a": "dentro_rango_a", "dentro_b": "dentro_rango_TN"}))


conteos = conteo(comparacion["subconjunto común"]).merge(
    conteo(comparacion["test completo"])[["indice_original", "dentro_rango_a", "dentro_rango_TN"]],
    on="indice_original", suffixes=("", "_test_completo"))
conteos = conteos.merge(errores[["indice_original", "probabilidad"]], on="indice_original")
conteos = conteos.sort_values(["resultado", "tipo", "probabilidad"], ignore_index=True)
conteos.insert(4, "features_evaluadas", len(TOP))

print(f"Cantidad de las {len(TOP)} features dentro de P5–P95 (percentiles del subconjunto común; "
      "columnas *_test_completo con percentiles del test completo):")
print(conteos.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
cambian = conteos[(conteos["dentro_rango_a"] != conteos["dentro_rango_a_test_completo"]) |
                  (conteos["dentro_rango_TN"] != conteos["dentro_rango_TN_test_completo"])]
print(f"\nErrores cuyo conteo cambia al usar percentiles del test completo: {len(cambian)}")

agregado = conteos.groupby("resultado").agg(
    errores=("indice_original", "size"),
    todas_dentro_a=("dentro_rango_a", lambda s: int((s == len(TOP)).sum())),
    todas_dentro_TN=("dentro_rango_TN", lambda s: int((s == len(TOP)).sum())),
    mas_dentro_TN_que_a=("dentro_rango_TN", lambda s: int((s > conteos.loc[s.index, "dentro_rango_a"]).sum())),
    mas_dentro_a_que_TN=("dentro_rango_a", lambda s: int((s > conteos.loc[s.index, "dentro_rango_TN"]).sum())),
)
print("\nConteo agregado por tipo de error (solo descriptivo):")
print(agregado.to_string())

guardar(comparacion["subconjunto común"], "comparacion_rangos_por_feature")
guardar(comparacion["test completo"], "comparacion_rangos_por_feature_test_completo")
guardar(conteos, "comparacion_rangos_conteo")
guardar(agregado.reset_index(), "comparacion_rangos_agregado")

Cantidad de las 12 features dentro de P5–P95 (percentiles del subconjunto común; columnas *_test_completo con percentiles del test completo):
 indice_original resultado         tipo          grupo_a  features_evaluadas  dentro_rango_a  dentro_rango_TN  dentro_rango_a_test_completo  dentro_rango_TN_test_completo  probabilidad
          119790        FN Slowhttptest  TP Slowhttptest                  12              10                9                            10                              9        0.0042
          119789        FN Slowhttptest  TP Slowhttptest                  12              10                8                            10                              8        0.0105
          116669        FN Slowhttptest  TP Slowhttptest                  12              12               12                            12                             12        0.3268
           82946        FN    Slowloris     TP Slowloris                  12               9                8         

In [14]:
# Detalle por feature: "<" debajo de P5, "=" dentro, ">" encima de P95 (subconjunto común)
SIMBOLO = {"debajo de P5": "<", "dentro": "=", "encima de P95": ">"}
c = comparacion["subconjunto común"]
for col, titulo in [("posicion_a", "(a) ataques detectados del mismo tipo"), ("posicion_b_TN", "(b) verdaderos negativos")]:
    detalle = c.assign(s=c[col].map(SIMBOLO)).pivot(index=["resultado", "tipo", "indice_original"], columns="feature",
                                                     values="s")[TOP]
    print(f"\nPosición frente a {titulo}:")
    with pd.option_context("display.width", 300, "display.max_columns", 40):
        print(detalle.to_string())


Posición frente a (a) ataques detectados del mismo tipo:
feature                                Bwd Packet Length Mean Fwd IAT Total Total Length of Fwd Packets Fwd IAT Mean Fwd Packet Length Min min_seg_size_forward PSH Flag Count Idle Max Fwd Packet Length Max Packet Length Variance Fwd IAT Max Fwd Packet Length Mean
resultado tipo         indice_original                                                                                                                                                                                                                                 
FN        Slowhttptest 116669                               =             =                           =            =                     =                    =              =        =                     =                      =           =                      =
                       119789                               =             =                           >            =                     =            

## 5. Duplicados exactos con train

Se usa el mismo hash que `reportar_duplicados_de_features` de `src/preprocessing/clean.py`: `pd.util.hash_pandas_object` sobre las 55 features, sin índice. Primero se llama a la función tal cual, como en el 01, y luego se aplica el mismo hash fila por fila para marcar cada flujo de test. Como control independiente del tipo de dato, se compara además con la igualdad exacta de los vectores en float64.

> **Nota.** "Duplicado" significa que las 55 features son idénticas. Los vectores casi iguales no se detectan con este criterio, así que este control no prueba que el modelo no memorice.

In [15]:
reporte_duplicados = reportar_duplicados_de_features(
    X_train, X_val, X_test, pd.Series(y_train), pd.Series(y_val), pd.Series(y_test)
)
guardar(reporte_duplicados, "reporte_duplicados")

# Mismo hash que el _hash interno de reportar_duplicados_de_features
columnas_hash = [c for c in X_train.columns if c not in ("Timestamp", "bloque")]
assert len(columnas_hash) == len(COLUMNAS)


def _hash(X: pd.DataFrame) -> pd.Series:
    return pd.util.hash_pandas_object(X[columnas_hash], index=False)


en_train = _hash(X_test).isin(set(_hash(X_train).to_numpy())).to_numpy()

tuplas_train = set(map(tuple, X_train[columnas_hash].to_numpy(dtype=np.float64)))
en_train_tuplas = np.fromiter((t in tuplas_train for t in map(tuple, X_test[columnas_hash].to_numpy(dtype=np.float64))),
                              dtype=bool, count=len(X_test))
assert np.array_equal(en_train, en_train_tuplas), "El hash y la igualdad exacta en float64 no coinciden"
ataques_en_train_reporte = int(reporte_duplicados.query("split == 'test' and clase == 1")["en_train"].iloc[0])
assert int((en_train & (y_test == 1)).sum()) == ataques_en_train_reporte, "El marcado fila por fila no coincide con el reporte"
print(f"\nHash fila por fila = igualdad exacta en float64 = reportar_duplicados_de_features "
      f"({ataques_en_train_reporte} ataques de test con vector en train).")

split  clase  filas  dup_en_split  en_train
train      0 302299         29556       NaN
train      1   8230           288       NaN
  val      0  64657          4807    7848.0
  val      1   1884           180     121.0
 test      0  65369          4142    8084.0
 test      1   1175           109      33.0

Filas cuyo vector aparece con ambas etiquetas: 0
Tabla guardada en reports\xgboost\analisis_errores\reporte_duplicados.csv



Hash fila por fila = igualdad exacta en float64 = reportar_duplicados_de_features (33 ataques de test con vector en train).


In [16]:
dup_errores = errores[["indice_original", "resultado", "tipo", "probabilidad"]].copy()
dup_errores["vector_en_train"] = pd.Series(en_train, index=indice_test).loc[dup_errores["indice_original"]].to_numpy()
n_fn_train = int(dup_errores.loc[dup_errores["resultado"] == "FN", "vector_en_train"].sum())
n_fp_train = int(dup_errores.loc[dup_errores["resultado"] == "FP", "vector_en_train"].sum())
print("Vector exacto de cada error presente en train:")
print(dup_errores.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nFN con vector en train: {n_fn_train} de {int(es_fn.sum())} | FP con vector en train: {n_fp_train} de {int(es_fp.sum())}")
guardar(dup_errores, "duplicados_errores")

Vector exacto de cada error presente en train:
 indice_original resultado         tipo  probabilidad  vector_en_train
          119790        FN Slowhttptest        0.0042            False
          119789        FN Slowhttptest        0.0105            False
          116669        FN Slowhttptest        0.3268            False
           82946        FN    Slowloris        0.0000            False
           82962        FN    Slowloris        0.0001            False
           82965        FN    Slowloris        0.0067            False
           82963        FN    Slowloris        0.0071            False
           82959        FN    Slowloris        0.0073            False
           82964        FN    Slowloris        0.0239            False
           82819        FN    Slowloris        0.8608            False
           82821        FN    Slowloris        0.8608            False
          396766        FP      benigno        0.9972            False
          116965        FP    

In [17]:
ataque = y_test == 1
filas = []
for ambito, m_amb in AMBITOS.items():
    for grupo, m_g in [("vector en train", en_train), ("vector no presente en train", ~en_train),
                       ("todos los ataques", np.ones_like(en_train))]:
        m = ataque & m_amb & m_g
        n, det = int(m.sum()), int((m & (y_pred == 1)).sum())
        fila = {"ambito": ambito, "grupo": grupo, "ataques": n, "detectados": det, "FN": n - det,
                "recall": det / n if n else np.nan}
        for t in TIPOS_ATAQUE:
            mt = m & (tipo == t)
            fila[f"detectados/ataques {t}"] = f"{int((mt & (y_pred == 1)).sum())}/{int(mt.sum())}"
        filas.append(fila)
recall_duplicados = pd.DataFrame(filas)
print(f"Recall a umbral {UMBRAL_FINAL} según si el vector exacto del ataque está en train:")
with pd.option_context("display.width", 250):
    print(recall_duplicados.to_string(index=False, float_format=lambda v: f"{v:.6f}"))

sin_dup = recall_duplicados.pivot(index="ambito", columns="grupo", values="recall")
sin_dup = sin_dup.rename(columns={"todos los ataques": "recall con todos los ataques",
                                  "vector no presente en train": "recall excluyendo vectores en train"})
sin_dup = sin_dup[["recall con todos los ataques", "recall excluyendo vectores en train"]]
sin_dup["diferencia"] = sin_dup["recall excluyendo vectores en train"] - sin_dup["recall con todos los ataques"]
print("\nRecall con y sin los ataques cuyo vector está en train:")
print(sin_dup.to_string(float_format=lambda v: f"{v:.6f}"))

print(f"\nAtaques de test con vector en train, por tipo y bloque (test completo):")
print(pd.crosstab(pd.Series(tipo[ataque & en_train], name="tipo"),
                  pd.Series(bloque_test[ataque & en_train], name="bloque")).to_string())
guardar(recall_duplicados, "recall_por_duplicado")
guardar(sin_dup.reset_index(), "recall_sin_duplicados")

Recall a umbral 0.87 según si el vector exacto del ataque está en train:
           ambito                       grupo  ataques  detectados  FN   recall detectados/ataques Slowloris detectados/ataques Slowhttptest
subconjunto común             vector en train       33          33   0 1.000000                        33/33                             0/0
subconjunto común vector no presente en train     1131        1120  11 0.990274                      873/881                         247/250
subconjunto común           todos los ataques     1164        1153  11 0.990550                      906/914                         247/250
    test completo             vector en train       33          33   0 1.000000                        33/33                             0/0
    test completo vector no presente en train     1142        1131  11 0.990368                      874/882                         257/260
    test completo           todos los ataques     1175        1164  11 0.990638  

## 6. Flujos cerca del umbral

Todos los flujos de test cuya probabilidad está a 0.1 o menos del umbral, por encima o por debajo, sin importar si están bien o mal clasificados.

In [18]:
cerca = np.abs(proba - UMBRAL_FINAL) <= VENTANA_UMBRAL + 1e-12
tabla_cerca = tabla_flujos(cerca, []).sort_values("probabilidad", ignore_index=True)
tabla_cerca["distancia_al_umbral"] = tabla_cerca["probabilidad"] - UMBRAL_FINAL
print(f"Flujos con probabilidad en [{UMBRAL_FINAL - VENTANA_UMBRAL:.2f}, {UMBRAL_FINAL + VENTANA_UMBRAL:.2f}]:")
print(pd.crosstab(pd.Series(resultado[cerca], name="resultado"),
                  pd.Series(np.where(mascara_comun[cerca], "subconjunto común", "fuera del común"), name="ámbito"),
                  margins=True, margins_name="total").to_string())
print()
with pd.option_context("display.width", 250, "display.max_rows", 500):
    print(tabla_cerca.to_string(float_format=lambda v: f"{v:.4f}"))
guardar(tabla_cerca, "cerca_del_umbral")

Flujos con probabilidad en [0.77, 0.97]:
ámbito     subconjunto común  total
resultado                          
FN                         2      2
TP                        29     29
total                     31     31

    indice_original     etiqueta_real          tipo resultado  probabilidad  bloque  en_subconjunto_comun  distancia_al_umbral
0             82819     DoS slowloris     Slowloris        FN        0.8608       2                  True              -0.0092
1             82821     DoS slowloris     Slowloris        FN        0.8608       2                  True              -0.0092
2            119730  DoS Slowhttptest  Slowhttptest        TP        0.8779       4                  True               0.0079
3            119779  DoS Slowhttptest  Slowhttptest        TP        0.8779       4                  True               0.0079
4             82822     DoS slowloris     Slowloris        TP        0.8970       2                  True               0.0270
5            119

## 7. Bloques temporales de los errores

Distribución por bloque temporal de los FN y los FP. Como referencia, también se incluyen todos los ataques de test por bloque.

In [19]:
fn_bloque = pd.crosstab(pd.Series(tipo[es_fn], name="tipo"), pd.Series(bloque_test[es_fn], name="bloque"))
fp_bloque = pd.Series(bloque_test[es_fp]).value_counts().sort_index().rename_axis("bloque").rename("FP")
ataques_bloque = pd.crosstab(pd.Series(tipo[ataque], name="tipo"), pd.Series(bloque_test[ataque], name="bloque"))
print("FN por tipo y bloque:")
print(fn_bloque.to_string())
print("\nFP por bloque:")
print(fp_bloque.to_string())
print("\nTodos los ataques de test por tipo y bloque (referencia):")
print(ataques_bloque.to_string())
guardar(fn_bloque.rename_axis(columns=None).reset_index(), "falsos_negativos_por_bloque")
guardar(fp_bloque.reset_index(), "falsos_positivos_por_bloque")

FN por tipo y bloque:
bloque        2  4
tipo              
Slowhttptest  0  3
Slowloris     8  0

FP por bloque:
bloque
1    2
4    2
5    4

Todos los ataques de test por tipo y bloque (referencia):
bloque          2    4
tipo                  
Slowhttptest    0  260
Slowloris     915    0
Tabla guardada en reports\xgboost\analisis_errores\falsos_negativos_por_bloque.csv
Tabla guardada en reports\xgboost\analisis_errores\falsos_positivos_por_bloque.csv


## Resumen

El resumen se genera en la celda siguiente a partir de los valores calculados en este cuaderno, para que ninguna cifra quede escrita a mano. Separa los hechos medidos de la interpretación.

In [20]:
mc_c, mc_t = matriz_confusion.loc["subconjunto común"], matriz_confusion.loc["test completo"]
fn_tipo = fn["tipo"].value_counts().reindex(TIPOS_ATAQUE, fill_value=0)
n_umbral = int((fn["categoria"] == "error de umbral").sum())
n_sin_senal = int((fn["categoria"] == "error sin señal").sum())
probs_fn = ", ".join(f"{p:.4f}" for p in sorted(fn["probabilidad"]))
pc = payload.query("ambito == 'subconjunto común'").set_index("grupo")
pt = payload.query("ambito == 'test completo'").set_index("grupo")


def prop(grupo: str, tabla: pd.DataFrame = pc) -> str:
    f = tabla.loc[grupo]
    return f"{int(f['payload_fwd_cero']):,}/{int(f['flujos']):,} ({f['proporcion']:.2%})"


rd = recall_duplicados.set_index(["ambito", "grupo"])


def rec(ambito: str, grupo: str) -> str:
    f = rd.loc[(ambito, grupo)]
    return f"{int(f['detectados']):,}/{int(f['ataques']):,} = {f['recall']:.6f}"


ag = agregado
fn_bloque_txt = "; ".join(f"{t}: " + ", ".join(f"bloque {b}: {int(n)}" for b, n in fn_bloque.loc[t].items() if n > 0)
                          for t in fn_bloque.index)
fp_bloque_txt = ", ".join(f"bloque {b}: {int(n)}" for b, n in fp_bloque.items())
cerca_res = pd.Series(resultado[cerca]).value_counts()
cerca_txt = ", ".join(f"{k} {int(v)}" for k, v in cerca_res.items())
n_tn_mas_ancho = int((anchos["TN"] > anchos["TP (ambos tipos)"]).sum())
dif_t = sin_dup.loc["test completo", "diferencia"]
dif_c = sin_dup.loc["subconjunto común", "diferencia"]

texto = f"""
### (a) Hechos medidos en este cuaderno

- **Matriz de confusión a umbral {UMBRAL_FINAL}.** Subconjunto común ({int(mc_c['flujos']):,} flujos: {int(mc_c['ataques']):,} ataques y {int(mc_c['benignos']):,} benignos): TN {int(mc_c['TN']):,}, FP {int(mc_c['FP'])}, FN {int(mc_c['FN'])}, TP {int(mc_c['TP']):,}. Test completo ({int(mc_t['flujos']):,} flujos): TN {int(mc_t['TN']):,}, FP {int(mc_t['FP'])}, FN {int(mc_t['FN'])}, TP {int(mc_t['TP']):,}. Ambas coinciden con `resultados_test.json`. Los {len(errores)} errores están {"todos dentro del" if errores['en_subconjunto_comun'].all() else "no todos dentro del"} subconjunto común.
- **Falsos negativos ({int(es_fn.sum())}).** {', '.join(f'{int(n)} {t}' for t, n in fn_tipo.items())}. Probabilidades: {probs_fn}. Con la definición operativa (0.1 o menos por debajo del umbral), hay {n_umbral} errores de umbral y {n_sin_senal} errores sin señal.
- **Payload forward igual a cero (subconjunto común).** FN: {prop('FN (ambos tipos)')}; Slowloris detectados: {prop('TP Slowloris')}; Slowhttptest detectados: {prop('TP Slowhttptest')}; verdaderos negativos: {prop('TN')}. En el test completo: Slowloris detectados {prop('TP Slowloris', pt)}, Slowhttptest detectados {prop('TP Slowhttptest', pt)}, verdaderos negativos {prop('TN', pt)}.
- **Falsos positivos ({int(es_fp.sum())}).** Probabilidad de {fp['probabilidad'].min():.4f} a {fp['probabilidad'].max():.4f}. Total Fwd Packets de {int(fp['Total Fwd Packets'].min())} a {int(fp['Total Fwd Packets'].max())}; Total Backward Packets de {int(fp['Total Backward Packets'].min())} a {int(fp['Total Backward Packets'].max())}. Flow Duration de {fp['Flow Duration'].min():,.0f} a {fp['Flow Duration'].max():,.0f}. Payload forward igual a cero en {int((fp['Total Length of Fwd Packets'] == 0).sum())} de {len(fp)}. min_seg_size_forward con valores {sorted(fp['min_seg_size_forward'].unique().tolist())}; Init_Win_bytes_forward {sorted(fp['Init_Win_bytes_forward'].unique().tolist())}; Init_Win_bytes_backward {sorted(fp['Init_Win_bytes_backward'].unique().tolist())}. Por bloque: {fp_bloque_txt}.
- **Rangos P5–P95 de las {len(TOP)} features con mayor gain (conteo descriptivo).** FN con todas dentro del rango de los detectados de su tipo: {int(ag.loc['FN', 'todas_dentro_a'])} de {int(ag.loc['FN', 'errores'])}; dentro del rango de los TN: {int(ag.loc['FN', 'todas_dentro_TN'])}. FP con todas dentro del rango de los ataques detectados: {int(ag.loc['FP', 'todas_dentro_a'])} de {int(ag.loc['FP', 'errores'])}; dentro del rango de los TN: {int(ag.loc['FP', 'todas_dentro_TN'])}. Errores cuyo conteo cambia con percentiles del test completo: {len(cambian)}.
- **Duplicados exactos (55 features).** FN con vector en train: {n_fn_train} de {int(es_fn.sum())}. FP con vector en train: {n_fp_train} de {int(es_fp.sum())}. Recall en el test completo: vector en train {rec('test completo', 'vector en train')}; vector no presente en train {rec('test completo', 'vector no presente en train')}; todos {rec('test completo', 'todos los ataques')}. Recall en el subconjunto común: vector en train {rec('subconjunto común', 'vector en train')}; vector no presente en train {rec('subconjunto común', 'vector no presente en train')}; todos {rec('subconjunto común', 'todos los ataques')}. Excluir los vectores en train cambia el recall en {dif_t:+.6f} (test completo) y {dif_c:+.6f} (subconjunto común).
- **Cerca del umbral (±{VENTANA_UMBRAL}).** {int(cerca.sum())} flujos en el test completo: {cerca_txt}.
- **Bloques de los FN.** {fn_bloque_txt}.

### (b) Interpretación

- Los conteos P5–P95 solo describen dónde caen los valores. El rango P5–P95 de los TN es más ancho que el de los ataques detectados (ambos tipos, subconjunto común) en {n_tn_mas_ancho} de las {len(TOP)} features, y más angosto o igual en las {len(TOP) - n_tn_mas_ancho} restantes. En las features de rango amplio, estar dentro del rango no indica parecido. Estos conteos no se usan para atribuir causas a los errores.
- {"Ningún FN tiene su vector exacto en train. Por eso, la coincidencia exacta con train no aparece asociada a los ataques no detectados en este conjunto." if n_fn_train == 0 else f"{n_fn_train} FN tienen su vector exacto en train."} El criterio solo cubre vectores idénticos; no evalúa vectores casi iguales ni descarta memorización.
- La diferencia entre el recall con y sin los ataques cuyo vector está en train es de {abs(dif_t):.6f} en el test completo. Su magnitud depende de cuántos ataques tienen el vector en train ({int((ataque & en_train).sum())}) y no permite concluir nada sobre generalización fuera de esta partición.
- La separación entre errores de umbral y errores sin señal es operativa: describe la distancia de la probabilidad al umbral y no explica por qué el modelo asignó esa probabilidad. El umbral sigue congelado en {UMBRAL_FINAL}; este análisis no lo modifica.
"""
display(Markdown(texto))


### (a) Hechos medidos en este cuaderno

- **Matriz de confusión a umbral 0.87.** Subconjunto común (66,454 flujos: 1,164 ataques y 65,290 benignos): TN 65,282, FP 8, FN 11, TP 1,153. Test completo (66,544 flujos): TN 65,361, FP 8, FN 11, TP 1,164. Ambas coinciden con `resultados_test.json`. Los 19 errores están todos dentro del subconjunto común.
- **Falsos negativos (11).** 8 Slowloris, 3 Slowhttptest. Probabilidades: 0.0000, 0.0001, 0.0042, 0.0067, 0.0071, 0.0073, 0.0105, 0.0239, 0.3268, 0.8608, 0.8608. Con la definición operativa (0.1 o menos por debajo del umbral), hay 2 errores de umbral y 9 errores sin señal.
- **Payload forward igual a cero (subconjunto común).** FN: 6/11 (54.55%); Slowloris detectados: 284/906 (31.35%); Slowhttptest detectados: 56/247 (22.67%); verdaderos negativos: 7,080/65,282 (10.85%). En el test completo: Slowloris detectados 285/907 (31.42%), Slowhttptest detectados 65/257 (25.29%), verdaderos negativos 7,096/65,361 (10.86%).
- **Falsos positivos (8).** Probabilidad de 0.9972 a 0.9993. Total Fwd Packets de 6 a 7; Total Backward Packets de 0 a 0. Flow Duration de 31,066,281 a 63,147,176. Payload forward igual a cero en 8 de 8. min_seg_size_forward con valores [40]; Init_Win_bytes_forward [29200]; Init_Win_bytes_backward [-1]. Por bloque: bloque 1: 2, bloque 4: 2, bloque 5: 4.
- **Rangos P5–P95 de las 12 features con mayor gain (conteo descriptivo).** FN con todas dentro del rango de los detectados de su tipo: 6 de 11; dentro del rango de los TN: 4. FP con todas dentro del rango de los ataques detectados: 8 de 8; dentro del rango de los TN: 0. Errores cuyo conteo cambia con percentiles del test completo: 0.
- **Duplicados exactos (55 features).** FN con vector en train: 0 de 11. FP con vector en train: 0 de 8. Recall en el test completo: vector en train 33/33 = 1.000000; vector no presente en train 1,131/1,142 = 0.990368; todos 1,164/1,175 = 0.990638. Recall en el subconjunto común: vector en train 33/33 = 1.000000; vector no presente en train 1,120/1,131 = 0.990274; todos 1,153/1,164 = 0.990550. Excluir los vectores en train cambia el recall en -0.000271 (test completo) y -0.000276 (subconjunto común).
- **Cerca del umbral (±0.1).** 31 flujos en el test completo: TP 29, FN 2.
- **Bloques de los FN.** Slowhttptest: bloque 4: 3; Slowloris: bloque 2: 8.

### (b) Interpretación

- Los conteos P5–P95 solo describen dónde caen los valores. El rango P5–P95 de los TN es más ancho que el de los ataques detectados (ambos tipos, subconjunto común) en 3 de las 12 features, y más angosto o igual en las 9 restantes. En las features de rango amplio, estar dentro del rango no indica parecido. Estos conteos no se usan para atribuir causas a los errores.
- Ningún FN tiene su vector exacto en train. Por eso, la coincidencia exacta con train no aparece asociada a los ataques no detectados en este conjunto. El criterio solo cubre vectores idénticos; no evalúa vectores casi iguales ni descarta memorización.
- La diferencia entre el recall con y sin los ataques cuyo vector está en train es de 0.000271 en el test completo. Su magnitud depende de cuántos ataques tienen el vector en train (33) y no permite concluir nada sobre generalización fuera de esta partición.
- La separación entre errores de umbral y errores sin señal es operativa: describe la distancia de la probabilidad al umbral y no explica por qué el modelo asignó esa probabilidad. El umbral sigue congelado en 0.87; este análisis no lo modifica.
